# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library. All dataset elements are referenced by their unique `@id` fields to enable precise, reproducible analysis.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')  # Ignore DataFrame warnings for clean output

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Dataset Croissant id: {metadata.id}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List the available record sets and their fields by @id
print("Available Record Sets (by @id):\n")
record_sets = list(metadata.record_sets)
for i, rs in enumerate(record_sets):
    print(f"{i+1}. id: {rs.id}, name: {rs.name}")
    print("   Fields (@id):")
    for field in rs.fields:
        print(f"     - {field.id} ({field.name})")
    print()
# If only one record set is available, select its id for further steps
record_set_ids = [rs.id for rs in record_sets]
first_record_set_id = record_set_ids[0] if record_set_ids else None

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

*If only one record set is present, it will be used for extraction. All field names and columns referenced are by `@id`.*

In [ ]:
# Extract data from each record set (by @id)
dataframes = {}
if record_set_ids:
    for record_set_id in record_set_ids:
        print(f"Loading records for: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns (@id): {df.columns.tolist()}")
        print(df.head(2))
else:
    print("No record sets were found in the dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. All field references utilize `@id` values as shown above.

In [ ]:
# For demo: pick a numeric field (e.g., molecular weight, number of peptides, coverage, etc.)
df = dataframes[first_record_set_id]

# Try to pick candidate numeric fields from columns
numeric_candidates = [col for col in df.columns if df[col].dtype in [int, float] or pd.api.types.is_numeric_dtype(df[col])]
if not numeric_candidates:
    # Try to infer numeric by attempting to convert
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col], errors='ignore')
        except Exception:
            continue
    numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

print(f"Candidate numeric fields (@id): {numeric_candidates}")

if numeric_candidates:
    numeric_field = numeric_candidates[0]  # pick the first
    threshold = df[numeric_field].dropna().mean()  # use mean as threshold
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (by @id):")
    print(filtered_df[[numeric_field]].head())

    # Normalization
    mean = filtered_df[numeric_field].mean()
    std = filtered_df[numeric_field].std()
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - mean) / std
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    # Pick a group field (non-numeric)
    group_candidates = [col for col in df.columns if df[col].dtype == object and col != numeric_field]
    if group_candidates:
        group_field = group_candidates[0]
        print(f"\nGrouping by: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(grouped_df.head())
    else:
        print("No suitable field for grouping was found.")
else:
    print("No numeric fields were found in the dataset.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. All axes and labels reference `@id` values for reproducibility.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_candidates:
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.xlabel(numeric_field + " (@id)")
    plt.title(f"Distribution of {numeric_field} (@id)")
    plt.tight_layout()
    plt.show()
    if 'group_field' in locals():
        plt.figure(figsize=(7,4))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.xlabel(group_field + " (@id)")
        plt.ylabel(numeric_field + " (@id)")
        plt.title(f"{numeric_field} grouped by {group_field} (@id)")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to access and analyze a dataset defined by a Croissant schema using the `mlcroissant` library. All dataset entities, columns, and fields are referenced using their `@id` to ensure precise referencing. For deeper analysis or custom pipelines, refer to the Croissant schema and the [mlcroissant documentation](https://github.com/mlcommons/croissant) for additional guidance.